In [1]:
import h5py
import numpy as np


In [2]:
h5=h5py.File('../moist_adiab/reconstruct_temp_prof_moist_10000.h5', 'r') 
Temperature = h5['Temperature'][:]
print(Temperature.shape)
h5.close()
Ts=Temperature[:,:,0]

print(Ts.shape)

nstep=Ts.shape[0]
nwalkers=Ts.shape[1]
print(nstep, nwalkers)

(10000, 8, 1600)
(10000, 8)
10000 8


In [3]:
h5=h5py.File(f'redo_emcee_moistadiab_FabianoLD_parallel_10000.h5', 'r') 

chain = h5['mcmc']['chain'][:,:,:]
[nstep,nwalk,ndim]=chain.shape

h5.close()
print(chain.shape)
# flattened_chain = chain[5000:,:,:].reshape(-1,3)

(10000, 8, 3)


In [4]:


#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

from multiprocessing import Pool, current_process
import threading
import queue
import time

from tqdm import tqdm

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation


nx2 = 8  ## shall not be less than N_walkers, can be a little greater for safty.

## initialize Canoe
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")
# print(pin.get_real("problem", "qH2O.ppmv"))

pin.set_boolean("job","verbose", False)

print(pin.get_string("mesh","nx2"))
pin.set_string("mesh","nx2", f"{nx2}")

print(pin.get_string("mesh","nx2"))

# pin.set_Real("radiation","outdir","(0,) (15,) (30,) (45,)")
print(pin.get_string("radiation","outdir"))

# exit()

mesh = Mesh(pin)
mesh.initialize(pin)

global mb
mb = mesh.meshblock(0)


T1bar = np.zeros((nstep,nwalkers))

for istep in tqdm(range(nstep)):
    for iwalker in range(nwalk):
        qH2O, t0, rH= chain[istep,iwalker,:]
        
        T1bar[istep,iwalker]=mb.derive_T1bar_given_Ts(pin, 354.96, Ts[istep, iwalker], "dry", qH2O)

profOU = h5py.File(f'reconstruct_temp_to_deeptheta_{nstep}.h5', 'w')
profOU.create_dataset('PotentialTemperature', data=T1bar)
profOU.close()

5Log, "2025-09-23 16:17:33",        canoe, 1., "Installing monitor canoe"
Log, "2025-09-23 16:17:33",        canoe, 1.1., "Initialize IndexMap"

8
(0,) (15,) (30,) (45,)
Log, "2025-09-23 16:17:33",         snap, 3., "Installing monitor snap"
Log, "2025-09-23 16:17:33",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-23 16:17:33",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-23 16:17:33",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-23 16:17:33",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-23 16:17:33",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-23 16:17:33",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-23 16:17:33", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-23 16:17:33", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-23 16:17:33",         harp, 9., "Installing monitor harp"
Log, "2025-09-23 16:17:33",         harp, 9.1., "Initialize

  0%|          | 2/10000 [00:00<13:14, 12.58it/s]

Log, "2025-09-23 16:17:45", pycanoe_infer_T0, 18., "Installing monitor pycanoe_infer_T0"


100%|██████████| 10000/10000 [12:28<00:00, 13.36it/s]
